# ACELO Cluster Optimization

Analyses Spark cluster telemetry and produces rightsizing recommendations.

**Managed by ACELO.** Deployed into this workspace by ACELO provisioning and
overwritten on the next deployment — edit the source in ACELO
(`backend/optimization_package/cluster/`), not this copy.

**Read-only against customer data.** Reads the configured source table and
writes only to the configured ACELO result table. It never modifies customer
source tables and never changes cluster configuration.

Scoring, thresholds and the XGBoost models are unchanged from the original
`Clusterfabric.ipynb`; only the source/result/model/credential bindings were
made runtime-configurable. See `RESULT_SCHEMA.md`.


In [ ]:
# PARAMETERS CELL — values are injected by ACELO at run time.
# Tag this cell as "Parameters" in Fabric so the Job Scheduler can override them.
# Defaults are intentionally EMPTY: the notebook fails loudly rather than
# silently analysing the wrong table.

acelo_run_id = ""         # ACELO JobRun id — stamped on every result row
environment_id = ""       # ACELO environment id, for traceability
source_table = ""         # customer telemetry table to READ
result_table = ""         # ACELO-owned table to WRITE
source_lakehouse = ""     # optional; qualifies source_table when unqualified
result_lakehouse = ""     # optional; qualifies result_table when unqualified
model_dir = ""            # optional; where XGBoost models persist
column_mapping = ""       # optional JSON: {"source_column": "expected_column"}
llm_key_vault_uri = ""    # optional Azure Key Vault URI holding the LLM key
llm_secret_name = ""      # optional secret name inside that Key Vault
llm_model_name = ""       # optional LLM model override
source_schema = ""
result_schema = ""

In [ ]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Every identifier is supplied per run, so one deployed copy serves every
# ACELO customer. Nothing customer-specific is baked into this notebook.

import json as _json
from datetime import datetime, timezone


def _as_text(value):
    """
    Safely normalize Fabric notebook parameter values to Python strings.

    Fabric pipeline parameters can sometimes arrive through Py4J / Java-backed
    objects rather than native Python strings. Never call .strip() directly
    on an unknown runtime value.
    """
    if value is None:
        return ""

    if isinstance(value, str):
        return value.strip()

    # Try normal Java/Py4J conversion first.
    try:
        java_value = value.toString()
        return str(java_value).strip()
    except Exception:
        return str(value).strip()


def _qualify(table, lakehouse, schema=""):
    """
    Resolve the Spark table name.

    Already qualified:
        dbo.table
        Data.dbo.table
        workspace.lakehouse.schema.table

    Schema supplied:
        dbo.table

    Only lakehouse supplied:
        Data.table

    Nothing else:
        table
    """
    table = _as_text(table)
    lakehouse = _as_text(lakehouse)
    schema = _as_text(schema)

    if not table:
        return ""

    # If already qualified, preserve exactly what was supplied.
    if "." in table:
        return table

    # Schema-enabled Lakehouse uses the notebook's default Lakehouse.
    if schema:
        return f"{schema}.{table}"

    # Legacy/non-schema Lakehouse case.
    if lakehouse:
        return f"{lakehouse}.{table}"

    return table


# ------------------------------------------------------------------------------
# Resolve source/result tables
# IMPORTANT: schema MUST be passed here.
# ------------------------------------------------------------------------------

SOURCE_TABLE = _qualify(
    source_table,
    source_lakehouse,
    source_schema,
)

RESULT_TABLE = _qualify(
    result_table,
    result_lakehouse,
    result_schema,
)

ACELO_RUN_ID = _as_text(acelo_run_id)
ACELO_ENVIRONMENT_ID = _as_text(environment_id)
ACELO_ANALYZED_AT = datetime.now(timezone.utc).isoformat()


# ------------------------------------------------------------------------------
# Runtime parameter trace
# ------------------------------------------------------------------------------

ACELO_RECEIVED_PARAMETERS = {
    "acelo_run_id": _as_text(acelo_run_id),
    "environment_id": _as_text(environment_id),
    "source_table": _as_text(source_table),
    "result_table": _as_text(result_table),
    "source_lakehouse": _as_text(source_lakehouse),
    "result_lakehouse": _as_text(result_lakehouse),
    "source_schema": _as_text(source_schema),
    "result_schema": _as_text(result_schema),
    "column_mapping": _as_text(column_mapping),
    "model_dir": _as_text(model_dir),
    "llm_model_name": _as_text(llm_model_name),
}

# Only parameters ACELO actually sends are compared.
ACELO_RECEIVED_PARAMETERS = {
    k: v
    for k, v in ACELO_RECEIVED_PARAMETERS.items()
    if v
}


print("[NOTEBOOK_RUNTIME_PARAMETERS]")

for _name in (
    "acelo_run_id",
    "environment_id",
    "source_table",
    "result_table",
    "source_lakehouse",
    "result_lakehouse",
    "source_schema",
    "result_schema",
    "column_mapping",
    "model_dir",
    "llm_model_name",
):
    print(
        f"{_name}={ACELO_RECEIVED_PARAMETERS.get(_name, '')}"
    )

print(
    f"llm_key_vault_uri="
    f"{'<set>' if _as_text(llm_key_vault_uri) else '<unset>'}"
)

print(
    f"llm_secret_name="
    f"{'<set>' if _as_text(llm_secret_name) else '<unset>'}"
)


# ------------------------------------------------------------------------------
# Required parameter validation
# ------------------------------------------------------------------------------

_missing_params = [
    name
    for name, value in [
        ("source_table", SOURCE_TABLE),
        ("result_table", RESULT_TABLE),
        ("acelo_run_id", ACELO_RUN_ID),
    ]
    if not value
]

if _missing_params:
    print("[NOTEBOOK_RUNTIME]")
    print(f"acelo_run_id={ACELO_RUN_ID}")
    print("parameter_validation=failed")
    print(
        f"missing_parameter={','.join(_missing_params)}"
    )

    raise ValueError(
        "ACELO Cluster Optimization is missing required parameters: "
        + ", ".join(_missing_params)
        + ". These are supplied by ACELO at run time; running this notebook "
        "interactively requires setting them in the Parameters cell."
    )


# ------------------------------------------------------------------------------
# Columns the optimizer reads from the source table
# ------------------------------------------------------------------------------

REQUIRED_COLUMNS = [
    "cluster_id",
    "cluster_name",
    "node_type",
    "current_workers",
    "min_workers",
    "max_workers",
    "avg_cpu_util",
    "avg_memory_util",
    "idle_time_min",
    "cluster_uptime_hours",
    "total_jobs_run",
    "total_dbus_cost_usd",
]


# ------------------------------------------------------------------------------
# Customer column -> optimizer column
# ------------------------------------------------------------------------------

DEFAULT_COLUMN_MAPPING = {
    "acme_cluster_id": "cluster_id",
}

COLUMN_MAPPING = dict(DEFAULT_COLUMN_MAPPING)

_column_mapping_text = _as_text(column_mapping)

if _column_mapping_text:
    try:
        COLUMN_MAPPING.update(
            _json.loads(_column_mapping_text)
        )
    except ValueError as exc:
        raise ValueError(
            "column_mapping must be a JSON object of "
            '{"source_column": "expected_column"} pairs.'
        ) from exc


# ------------------------------------------------------------------------------
# Runtime status
# ------------------------------------------------------------------------------

print("[NOTEBOOK_RUNTIME]")
print(f"acelo_run_id={ACELO_RUN_ID}")
print("notebook_started=true")
print("parameter_validation=passed")


# ------------------------------------------------------------------------------
# Data path trace
# ------------------------------------------------------------------------------

print("[NOTEBOOK_DATA_PATH]")
print(f"acelo_run_id={ACELO_RUN_ID}")
print(
    f"source_lakehouse={_as_text(source_lakehouse)}"
)
print(f"source_table={SOURCE_TABLE}")
print(
    f"result_lakehouse={_as_text(result_lakehouse)}"
)
print(f"result_table={RESULT_TABLE}")





# ------------------------------------------------------------------------------
# Final summary
# ------------------------------------------------------------------------------

print(f"ACELO run {ACELO_RUN_ID}")
print(f"  source: {SOURCE_TABLE}")
print(f"  result: {RESULT_TABLE}")

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import xgboost as xgb
from pyspark.sql.functions import col, when, lit, least, greatest, lower, coalesce

MODEL_NAME = llm_model_name or "llama-3.3-70b-versatile"

# LLM credential resolution.
#
# No API key is ever embedded in this notebook. The key is read from Azure Key
# Vault through Fabric's own secret API when the environment supplies a vault
# reference; failing that, from a GROQ_API_KEY environment variable if the
# workspace sets one.
#
# When no credential is available the deterministic optimization below still
# runs in full and the llm_optimization column records LLM_UNAVAILABLE. A
# recommendation is never fabricated.
LLM_UNAVAILABLE = "LLM_UNAVAILABLE"
_GROQ_API_KEY = None
_llm_status = LLM_UNAVAILABLE

if llm_key_vault_uri and llm_secret_name:
    try:
        try:
            import notebookutils
            _GROQ_API_KEY = notebookutils.credentials.getSecret(
                llm_key_vault_uri, llm_secret_name
            )
        except ImportError:
            from notebookutils import mssparkutils
            _GROQ_API_KEY = mssparkutils.credentials.getSecret(
                llm_key_vault_uri, llm_secret_name
            )
        _llm_status = "key_vault"
    except Exception as exc:
        # Deliberately does not include the exception text, which can echo the
        # secret URI.
        print(f"LLM credential could not be read from Key Vault ({type(exc).__name__}).")
else:
    _GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    if _GROQ_API_KEY:
        _llm_status = "environment"

LLM_ENABLED = bool(_GROQ_API_KEY)
print(f"LLM enrichment: {'enabled via ' + _llm_status if LLM_ENABLED else 'disabled (' + LLM_UNAVAILABLE + ')'}")


def call_groq_llm(prompt_text):
    if not LLM_ENABLED:
        return LLM_UNAVAILABLE

    try:
        from groq import Groq

        client = Groq(api_key=_GROQ_API_KEY)

        completion = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert FinOps Engine Optimization Agent."
                },
                {
                    "role": "user",
                    "content": prompt_text
                }
            ],
            temperature=0
        )

        return completion.choices[0].message.content.strip()

    except Exception as e:
        # Honest failure status, never an invented recommendation.
        return f"{LLM_UNAVAILABLE}: {type(e).__name__}"


# ---------------------------------------------------------------------------
# Load the customer's telemetry table (READ ONLY) and map its columns onto the
# names the optimizer uses. Nothing is written back to the source.
# ---------------------------------------------------------------------------
try:
    df = spark.read.table(SOURCE_TABLE)
except Exception as exc:
    print(f"[NOTEBOOK_DATA_READ] acelo_run_id={ACELO_RUN_ID} status=failed "
          f"source_table={SOURCE_TABLE} error={type(exc).__name__}")
    raise RuntimeError(
        f"ACELO could not read the configured source table '{SOURCE_TABLE}'. "
        "Check that it exists in this workspace and that the running identity "
        "can read it."
    ) from exc

# Rename customer columns onto the optimizer's expected names. Only applied
# where the source column actually exists and the target does not, so a table
# that already uses the expected names is untouched.
for _source_column, _expected_column in COLUMN_MAPPING.items():
    if _source_column in df.columns and _expected_column not in df.columns:
        df = df.withColumnRenamed(_source_column, _expected_column)

# Fail loudly on a genuine mismatch rather than inventing data.
_missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if _missing:
    raise ValueError(
        f"Source table '{SOURCE_TABLE}' is missing required columns: "
        + ", ".join(_missing)
        + ". Available columns: " + ", ".join(sorted(df.columns))
        + ". Map them with the column_mapping parameter, or supply a view that "
        "exposes them. ACELO does not substitute values for missing telemetry."
    )

_source_rows = df.count()
if _source_rows == 0:
    print(f"[NOTEBOOK_DATA_READ] acelo_run_id={ACELO_RUN_ID} status=failed "
          f"source_table={SOURCE_TABLE} reason=empty")
    raise ValueError(
        f"Source table '{SOURCE_TABLE}' contains no rows, so there is nothing to analyse."
    )

print(f"[NOTEBOOK_DATA_READ] acelo_run_id={ACELO_RUN_ID} status=success "
      f"source_table={SOURCE_TABLE} rows={_source_rows}")

df = df.fillna({
    "avg_cpu_util": 0,
    "total_jobs_run": 0,
    "cluster_uptime_hours": 0
})

df = df.withColumn(
    "current_workers",
    greatest(
        coalesce(col("current_workers"), lit(1)),
        lit(1)
    )
)

df_rates = df.select(
    (
        col("total_dbus_cost_usd") /
        greatest(col("cluster_uptime_hours"), lit(1))
    ).alias("cost_ph"),
    (
        col("total_jobs_run") /
        greatest(col("cluster_uptime_hours"), lit(1))
    ).alias("jobs_ph"),
    (
        col("total_jobs_run") /
        greatest(col("current_workers"), lit(1))
    ).alias("worker_util")
)

outlier_limits = df_rates.approxQuantile(
    ["cost_ph", "jobs_ph", "worker_util"],
    [0.95],
    0.01
)

max_cost_ph = max(outlier_limits[0][0], 1.0)
max_jobs_ph = max(outlier_limits[1][0], 1.0)
max_worker_util = max(outlier_limits[2][0], 1.0)

df = (
    df
    .withColumn(
        "cpu_norm",
        least(col("avg_cpu_util") / 100, lit(1.0))
    )
    .withColumn(
        "memory_norm",
        least(col("avg_memory_util") / 100, lit(1.0))
    )
    .withColumn(
        "idle_norm",
        least(
            col("idle_time_min") /
            greatest(
                col("cluster_uptime_hours") * 60,
                lit(1)
            ),
            lit(1.0)
        )
    )
    .withColumn(
        "cost_norm",
        least(
            (
                col("total_dbus_cost_usd") /
                greatest(col("cluster_uptime_hours"), lit(1))
            ) / lit(max_cost_ph),
            lit(1.0)
        )
    )
    .withColumn(
        "jobs_ph_norm",
        least(
            (
                col("total_jobs_run") /
                greatest(col("cluster_uptime_hours"), lit(1))
            ) / lit(max_jobs_ph),
            lit(1.0)
        )
    )
    .withColumn(
        "worker_util_norm",
        least(
            (
                col("total_jobs_run") /
                greatest(col("current_workers"), lit(1))
            ) / lit(max_worker_util),
            lit(1.0)
        )
    )
)

df = (
    df
    .withColumn(
        "idle_score",
        when(
            col("cluster_uptime_hours") > 1,
            (1 - col("cpu_norm")) * 0.30 +
            (1 - col("memory_norm")) * 0.25 +
            col("idle_norm") * 0.25 +
            (1 - col("jobs_ph_norm")) * 0.20
        ).otherwise(
            (1 - col("cpu_norm")) * 0.45 +
            (1 - col("memory_norm")) * 0.25 +
            col("idle_norm") * 0.10 +
            (1 - col("jobs_ph_norm")) * 0.20
        )
    )
    .withColumn(
        "idle_impact_score",
        col("idle_score") * 0.7 +
        col("cost_norm") * 0.3
    )
    .withColumn(
        "oversized_score",
        (1 - col("cpu_norm")) * 0.35 +
        (1 - col("memory_norm")) * 0.35 +
        (1 - col("worker_util_norm")) * 0.30
    )
)

total_clusters = df.count()

if total_clusters < 10:
    raw_idle_p75, raw_idle_p90 = 0.60, 0.75
    raw_over_p75, raw_over_p90 = 0.60, 0.75
else:
    quantiles = df.approxQuantile(
        ["idle_impact_score", "oversized_score"],
        [0.75, 0.90],
        0.01
    )

    raw_idle_p75, raw_idle_p90 = (
        quantiles[0][0],
        quantiles[0][1]
    )

    raw_over_p75, raw_over_p90 = (
        quantiles[1][0],
        quantiles[1][1]
    )

mod_idle_thresh = min(
    max(raw_idle_p75, 0.50),
    0.75
)

high_idle_thresh = max(
    min(max(raw_idle_p90, 0.65), 0.85),
    mod_idle_thresh + 0.05
)

mod_over_thresh = min(
    max(raw_over_p75, 0.50),
    0.75
)

high_over_thresh = max(
    min(max(raw_over_p90, 0.65), 0.85),
    mod_over_thresh + 0.05
)

specialized_workloads = "stream|etl|burst|continuous"

df = (
    df
    .withColumn(
        "underutilized_label",
        when(
            col("cluster_uptime_hours") <= 1,
            "Evaluating (New)"
        )
        .when(
            lower(col("cluster_name")).rlike(
                specialized_workloads
            ),
            "Specialized Workload"
        )
        .when(
            col("idle_impact_score") > lit(high_idle_thresh),
            "Highly Underutilized"
        )
        .when(
            col("idle_impact_score") > lit(mod_idle_thresh),
            "Moderately Underutilized"
        )
        .otherwise("Well Utilized")
    )
    .withColumn(
        "idle_flag",
        when(
            col("underutilized_label").isin(
                "Highly Underutilized",
                "Moderately Underutilized"
            ),
            1
        ).otherwise(0)
    )
)

df = (
    df
    .withColumn(
        "oversized_label",
        when(
            col("cluster_uptime_hours") <= 1,
            "Evaluating (New)"
        )
        .when(
            lower(col("cluster_name")).rlike(
                specialized_workloads
            ),
            "Specialized Workload"
        )
        .when(
            col("oversized_score") > lit(high_over_thresh),
            "Highly Oversized"
        )
        .when(
            col("oversized_score") > lit(mod_over_thresh),
            "Moderately Oversized"
        )
        .otherwise("Optimal")
    )
    .withColumn(
        "oversized_flag",
        when(
            col("oversized_label").isin(
                "Highly Oversized",
                "Moderately Oversized"
            ),
            1
        ).otherwise(0)
    )
)

df = df.fillna({
    "idle_flag": 0,
    "oversized_flag": 0
})

df = df.withColumn(
    "job_density_norm",
    least(
        col("total_jobs_run") /
        greatest(
            col("cluster_uptime_hours") *
            col("current_workers"),
            lit(1)
        ) / 2.0,
        lit(1.0)
    )
)

df = df.withColumn(
    "efficiency_score",
    ((1 - col("idle_flag")) * 0.3) +
    ((1 - col("oversized_flag")) * 0.3) +
    (col("job_density_norm") * 0.2) +
    ((1 - (col("avg_cpu_util") / 100)) * 0.2)
)

df = df.withColumn(
    "optimization_label",
    when(
        col("efficiency_score") > 0.70,
        "Optimized"
    )
    .when(
        col("efficiency_score") >= 0.40,
        "Moderately Optimized"
    )
    .otherwise("Risky")
)

df = df.withColumn(
    "recommended_max_workers",
    when(
        col("optimization_label") == "Risky",
        greatest(
            col("min_workers"),
            (col("current_workers") * 0.7).cast("int")
        )
    )
    .when(
        col("optimization_label") == "Moderately Optimized",
        greatest(
            col("min_workers"),
            (col("current_workers") * 0.9).cast("int")
        )
    )
    .otherwise(col("max_workers"))
)

df = df.withColumn(
    "recommended_max_workers",
    greatest(
        col("recommended_max_workers"),
        col("min_workers"),
        lit(1)
    )
)

pdf_inference = df.toPandas()

WH_ORDER = {
    "2X-Small": 0,
    "Small": 1,
    "Medium": 2,
    "Large": 3,
    "X-Large": 4,
    "2X-Large": 5
}

if "warehouse_size" in pdf_inference.columns:
    pdf_inference["warehouse_size_enc"] = (
        pdf_inference["warehouse_size"]
        .map(WH_ORDER)
        .fillna(0)
    )
else:
    pdf_inference["warehouse_size_enc"] = (
        pdf_inference["node_type"]
        .apply(
            lambda x:
                1 if "DS3" in str(x)
                else (
                    2 if "DS4" in str(x)
                    else (
                        3 if "DS5" in str(x)
                        else 0
                    )
                )
        )
    )

pdf_inference["cluster_id_enc"] = pd.factorize(
    pdf_inference["cluster_id"]
)[0]

pdf_inference["exec_norm"] = (
    pdf_inference["worker_util_norm"]
    .fillna(0)
)

pdf_inference["scan_norm"] = (
    pdf_inference["jobs_ph_norm"]
    .fillna(0)
)

pdf_inference["spill_norm"] = (
    1.0 - pdf_inference["memory_norm"]
).fillna(0)

pdf_inference["shuffle_norm"] = (
    1.0 - pdf_inference["cpu_norm"]
).fillna(0)

pdf_inference["cost_score_norm"] = (
    pdf_inference["cost_norm"]
    .fillna(0)
)

pdf_inference["wh_multiplier"] = (
    pdf_inference["current_workers"]
    .fillna(1)
    .astype(float)
)

FEATURES = [
    "exec_norm",
    "cpu_norm",
    "scan_norm",
    "spill_norm",
    "shuffle_norm",
    "cost_score_norm",
    "wh_multiplier",
    "warehouse_size_enc",
    "cluster_id_enc"
]

# Model persistence. Defaults to the attached Lakehouse Files area so models
# survive between runs; falls back to session-local storage when no lakehouse is
# mounted, in which case the boosters are retrained for this run.
MODEL_DIR = (model_dir or "").strip() or "/lakehouse/default/Files/models/finops"

try:
    os.makedirs(MODEL_DIR, exist_ok=True)
except Exception:
    MODEL_DIR = "/tmp/acelo/models/finops"
    os.makedirs(MODEL_DIR, exist_ok=True)
    print(f"No writable lakehouse path; using {MODEL_DIR} for this run.")

COST_MODEL_PATH = f"{MODEL_DIR}/xgb_cost_model.json"
SAVINGS_MODEL_PATH = f"{MODEL_DIR}/xgb_savings_model.json"

if not os.path.exists(COST_MODEL_PATH) or not os.path.exists(
    SAVINGS_MODEL_PATH
):
    TARGET_COST = "total_dbus_cost_usd"
    TARGET_SAVINGS = "oversized_score"

    train_df = pdf_inference[
        FEATURES + [
            TARGET_COST,
            TARGET_SAVINGS
        ]
    ].fillna(0)

    X_train = train_df[FEATURES].astype(float)
    y_cost = train_df[TARGET_COST].astype(float)
    y_save = (
        train_df[TARGET_SAVINGS]
        * 100
    ).clip(0, 100).astype(float)

    model_cost_init = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        verbosity=0
    )

    model_cost_init.fit(
        X_train,
        y_cost
    )

    model_savings_init = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        verbosity=0
    )

    model_savings_init.fit(
        X_train,
        y_save
    )

    model_cost_init.save_model(
        COST_MODEL_PATH
    )

    model_savings_init.save_model(
        SAVINGS_MODEL_PATH
    )

model_cost = xgb.XGBRegressor()
model_cost.load_model(COST_MODEL_PATH)

model_savings = xgb.XGBRegressor()
model_savings.load_model(SAVINGS_MODEL_PATH)

X_mat = (
    pdf_inference[FEATURES]
    .fillna(0)
    .astype(float)
)

pdf_inference["predicted_cost_usd"] = (
    model_cost.predict(X_mat)
)

pdf_inference["predicted_savings_pct"] = (
    model_savings.predict(X_mat)
)

pdf_inference["ml_cost_variance"] = (
    pdf_inference["total_dbus_cost_usd"] -
    pdf_inference["predicted_cost_usd"]
)

pdf_inference["potential_monthly_savings"] = (
    pdf_inference["total_dbus_cost_usd"] *
    (
        pdf_inference["predicted_savings_pct"] /
        100.0
    )
).clip(lower=0.0)


def apply_predictive_llm_logic(row):
    if row["optimization_label"] != "Risky":
        return "Health Stable. No AI action required."

    prompt = f"""
Analyze this specific cluster deployment:

Cluster Instance Profile:
{row['cluster_name']} ({row['node_type']})

Configuration:
{row['current_workers']} workers active
Autoscale limits: {row['min_workers']}-{row['max_workers']}

Current Recommended Target Max Worker Limit:
{row['recommended_max_workers']}

Uptime:
{row['cluster_uptime_hours']} hours

Production Jobs:
{row['total_jobs_run']}

XGBOOST PREDICTIVE PERFORMANCE SIGNALS:

Predicted Baseline Cost Target:
${row['predicted_cost_usd']:.2f}/hr

Observed Waste Overhead Variance:
${row['ml_cost_variance']:.2f}/hr

Model-Driven Proportional Savings:
{row['predicted_savings_pct']:.1f}%

Projected Monthly Savings:
${row['potential_monthly_savings']:.2f}/month

RESOURCE LOAD DISTRIBUTIONS:

CPU Waste Factor:
{row['shuffle_norm']:.2f}

Observed Avg CPU:
{row['avg_cpu_util']}%

Memory Waste Factor:
{row['spill_norm']:.2f}

Observed Avg Memory:
{row['avg_memory_util']}%

Idle Duration Overhead:
{row['idle_norm']:.2f}

Idle Time:
{row['idle_time_min']} minutes

Generate a highly prescriptive remediation plan.

1. Validate or adjust the recommended Max Workers value
   ({row['recommended_max_workers']}) based on the model signals.

2. Recommend downsized compute nodes or confirm that the
   instance family matches workload constraints.

3. Define strict Auto-Termination timeout parameters
   matching the current idle footprint.
"""

    return call_groq_llm(prompt)


pdf_inference["llm_optimization"] = pdf_inference.apply(
    apply_predictive_llm_logic,
    axis=1
)

pdf_inference["potential_monthly_savings"] = (
    pdf_inference["potential_monthly_savings"]
    .fillna(0.0)
)

pdf_inference["llm_optimization"] = (
    pdf_inference["llm_optimization"]
    .fillna("Health Stable. No AI action required.")
)

# ---------------------------------------------------------------------------
# Persist results to the ACELO-owned result table.
#
# Two deliberate differences from the original, both required for multi-run,
# multi-customer operation:
#
#   1. Every row is stamped with acelo_run_id, so ACELO retrieves exactly the
#      rows this run produced and never shows a stale result.
#   2. The write APPENDS rather than replacing the table, preserving run
#      history. The original replaced the whole table on every run.
#
# Only the ACELO result table is written. The customer's source table is never
# modified, and no cluster configuration is changed — this is analysis only.
# ---------------------------------------------------------------------------
pdf_inference["acelo_run_id"] = ACELO_RUN_ID
pdf_inference["acelo_environment_id"] = ACELO_ENVIRONMENT_ID
pdf_inference["acelo_analyzed_at"] = ACELO_ANALYZED_AT
# The parameters this run actually received, so ACELO can compare them with
# what it sent when it reads the result rows back.
pdf_inference["acelo_received_parameters"] = json.dumps(ACELO_RECEIVED_PARAMETERS, sort_keys=True)

final_result_df = spark.createDataFrame(
    pdf_inference
)

try:
    final_result_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(RESULT_TABLE)
except Exception as exc:
    print(f"[NOTEBOOK_RESULT_WRITE] acelo_run_id={ACELO_RUN_ID} status=failed "
          f"result_table={RESULT_TABLE} error={type(exc).__name__}")
    raise

_rows_written = final_result_df.count()
print(f"[NOTEBOOK_RESULT_WRITE] acelo_run_id={ACELO_RUN_ID} status=success "
      f"result_table={RESULT_TABLE} rows={_rows_written}")
print(
    f"ACELO run {ACELO_RUN_ID}: wrote {_rows_written} cluster result rows "
    f"to {RESULT_TABLE}"
)

# exitValue is surfaced by ACELO's job status polling.
try:
    import notebookutils

    notebookutils.notebook.exit(
        json.dumps(
            {
                "acelo_run_id": ACELO_RUN_ID,
                "rows_written": _rows_written,
                "result_table": RESULT_TABLE,
                "llm_enabled": LLM_ENABLED,
                "received_parameters": ACELO_RECEIVED_PARAMETERS,
            }
        )
    )
except ImportError:
    pass